In [ ]:
!pip install -qU \
langchain \
langchain-groq \
langchain-core \
langchain-community \
pydantic \
python-dotenv \
ddgs


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import os
import json
from pydantic import BaseModel, Field, ValidationError
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_community.tools import DuckDuckGoSearchRun
from google.colab import userdata

/tmp/ipykernel_1072/743123178.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [ ]:
class TravelOutput(BaseModel):
    destination: str = Field(
        description="The travel destination (city or country)",
        min_length=2,
    )

    budget: float | None = Field(
        default=None,
        gt=0,
        description="The total travel budget. Use null if the user did not mention one. Never guess it.",
    )

    duration: str = Field(
        description="The duration of the trip (e.g., 5 days, 2 weeks)",
        min_length=2,
    )

    summary: str = Field(
        description="A brief summary of the travel plan. Keep it under 400 characters.",
        min_length=10,
        max_length=800,
    )

In [ ]:
llm=ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0.5,
    max_tokens=2048,
    reasoning_effort="medium",
    api_key=userdata.get("GROQ_API_KEY")
)

In [ ]:
Trevel_Agent =create_agent(
   model=llm,
    name="Travel Agent",
    system_prompt=(
        "You are a friendly and knowledgeable travel planner that helps users plan trips, "
        "suggest destinations, and create brief summaries of their journeys. "
        "Use tools when helpful (e.g., web search)"
        "Keep answers short and concise."
    ),
    response_format=TravelOutput,
   tools=[
        DuckDuckGoSearchRun(name="Search")],
)



In [ ]:
def print_fields(data):

    if isinstance(data, str):
        try:
            data = TravelOutput(**json.loads(data))
        except (json.JSONDecodeError, ValidationError):
            print("Raw output:", data)
            return

    budget = f"{data.budget:,.2f}" if data.budget is not None else "Not specified"

    print(f"Destination: {data.destination}")
    print(f"Budget: \n{budget}")
    print(f"Duration: \n{data.duration}")
    print(f"Summary: \n{data.summary}")

In [ ]:
async def main(prompt: str, retries: int = 2):
    for attempt in range(1, retries + 1):
        try:
            result = await Trevel_Agent.ainvoke(
                {"messages": [{"role": "user", "content": prompt}]}
            )
            print_fields(result["structured_response"])
            return
        except Exception as e:
            print(f"Attempt {attempt} failed: {e}")
    print("All attempts failed.")

await main ("Plan a 3-day trip to Alexandria, Egypt, with a budget of $200. Include the best places to visit, affordable restaurants, accommodation, and a simple daily itinerary. Keep the total cost within budget.")

Attempt 1 failed: DNSError: DNSError('error sending request for url (https://wt.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=Alexandria%20Egypt%20cheap%20restaurants%20local%20food%20affordable%20eating) > error resolving DNS > DNS error: no records found for Query { name: Name("wt.wikipedia.org.google.internal."), query_type: A, query_class: IN }')
Attempt 2 failed: DNSError: DNSError('error sending request for url (https://wt.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=Alexandria%20Egypt%20cheap%20restaurants%20affordable%20food%202024) > error resolving DNS > DNS error: no records found for Query { name: Name("wt.wikipedia.org.google.internal."), query_type: A, query_class: IN }')
All attempts failed.
